In [1]:
import numpy             as np
import matplotlib.pyplot as plt
import seaborn           as sns
import os

from pymatgen.io.vasp.outputs import Vasprun

sns.set_theme()

First we compute linear macroscopic average potential for bulk and each slab. Then we compute the band alignment for each slab.

The linear macroscopic average potential is computed as:

$echo -e "427\n{direction}\n{threshold}\n{iterations}" | ~/vaspkit.1.3.5/bin/vaspkit

In [ ]:
# Define name to reference folder which stores the energies of each slab
ref_folder = 'input/slabs/BiSeI-slab'

# Bulk

In [ ]:
bulk_folder = f'{ref_folder}/bulk'

linear_mac_pot = np.loadtxt(f'{bulk_folder}/MACROSCOPIC_AVERAGE.dat')

In [ ]:
# Load vasprun
vasprun = Vasprun(f'{bulk_folder}/vasprun.xml')

# Get band structure
band_structure = vasprun.get_band_structure()

# Get valence band maximum and band gap
valence_band_DFT = band_structure.get_vbm()["energy"]
band_gap         = band_structure.get_band_gap()["energy"]

In [ ]:
energy_section = linear_mac_pot[:, 1]

mac_avg_pot_bulk = np.mean(energy_section)

np.savetxt(f'{bulk_folder}/macroscopic_average_potential', [mac_avg_pot_bulk])

print(f'Mean value: {mac_avg_pot_bulk}')

In [ ]:
plt.plot(linear_mac_pot[:, 0], linear_mac_pot[:, 1], label='Linear average')

plt.savefig(f'{bulk_folder}/macroscopic_average_potential.eps', bbox_inches='tight')
plt.show()

# Slab

In [ ]:
# Iterate over slabs
for miller_index_str in os.listdir(ref_folder):
    # Define current folder
    slab_folder = f'{ref_folder}/{miller_index_str}'

    # Skip in case it is not a folder or it is the bulk folder
    if (not os.path.isdir(slab_folder)) or (miller_index_str == 'bulk'):
        continue
    
    linear_mac_pot = np.loadtxt(f'{slab_folder}/MACROSCOPIC_AVERAGE.dat')
    
    energy_section = linear_mac_pot[:, 1]

    #### We need to know which one corresponds to slab and which one to vacuum
    left_section  = int(0.25*len(energy_section))
    right_section = int(0.75*len(energy_section))

    mac_avg_pot_left  = energy_section[left_section]
    mac_avg_pot_right = energy_section[right_section]

    # Save the data
    np.savetxt(f'{slab_folder}/macroscopic_average_potential', [mac_avg_pot_left, mac_avg_pot_right])

    print(slab_folder)
    print(f'Mean values: {mac_avg_pot_left}, {mac_avg_pot_right}')
    
    plt.plot(linear_mac_pot[:, 0], linear_mac_pot[:, 1], label='Linear average')
    
    plt.plot(linear_mac_pot[left_section,  0], mac_avg_pot_left,  'o')
    plt.plot(linear_mac_pot[right_section, 0], mac_avg_pot_right, 'o')

    plt.savefig(f'{slab_folder}/macroscopic_average_potential.eps', bbox_inches='tight')
    plt.show()
    
    # Compute band alignments referred to vacuum
    valence_band    = valence_band_DFT + mac_avg_pot_left - mac_avg_pot_bulk + 0 - mac_avg_pot_right
    conduction_band = valence_band + band_gap

    # Save the data
    np.savetxt(f'{slab_folder}/valence_band',    [valence_band])
    np.savetxt(f'{slab_folder}/conduction_band', [conduction_band])

VBM = VMB_DFT - V_bulk + (V_slab - V_vacuum)